# Prior sensitivity of the Student-t primary model (continuous BMI, ≥ 1,000 reads)

Refits the primary model (`__ri_hc_t`) with each prior changed one at a time, at publication length (4 chains × 4,000 draws, chains in parallel), and compares every fit with the primary.

In [ ]:
# 1. Install the validated sampling environment
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "pymc==5.28.5", "arviz==0.23.4", "nutpie==0.16.11"])
print("Pinned environment installed.")


In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
%%writefile /content/ri_hc_engine.py
FAMILY = CONFIG["family"]
PROFILE = CONFIG["profile"]
STAGE = CONFIG["stage"]
MIN_READS = CONFIG.get("min_reads", 1000)
LOCAL_PATHS = CONFIG.get("local_paths")
SAMPLER_OVERRIDE = CONFIG.get("sampler_override")
PPC_DRAWS_OVERRIDE = CONFIG.get("ppc_draws_override")
PROMOTION_WAIVER = CONFIG.get("waiver")

import gc, hashlib, json, math, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

NOTEBOOK_VERSION = "ri-hc-celltype-intercept-v1"
SCORE_VARIANT = "within_component_logcpm_z_cap5"
BASE = Path("/content/drive/MyDrive/new_dysfunction")
PCA_DIR = BASE / ("bayesprism_initial_issue_verification_v1/publication_pca_tiered_qc_v1/"
                  "20260906T013040524557Z")
SENS_ROOT = BASE / "normalized_state_bhm_ri_hc_v1"
PRIMARY_LOWREAD_ROOT = BASE / "normalized_state_bhm_lowread_sensitivity_v1"
CELLTYPE_INTERCEPT_SD = 0.50
WIDENING = 1.37
HEADLINE = {
    "immune_coarse": [("CD4+ T CELLS", "Helper_Function_Signature"),
                      ("CD4+ T CELLS", "CD4_Metabolic_Flexibility_Signature"),
                      ("CD8_T_GENERAL", "CD8_Metabolic_Programming_Signature")],
    "immune_fine": [("TAM", "Iron_Handling_Signature"), ("TAM", "Lipid_Processing_Signature"),
                    ("TAM", "Tissue_Repair_Signature"), ("CD8_T_EXHAUSTED", "Metabolic_Imbalance_Signature")],
    "nonimmune": [],
}

FAMILY_SPECS = {
    "immune_coarse": {
        "compartment": "immune_course", "tier_column": "tiered_primary_keep",
        "nu_scope": "celltype", "architecture": "v2p2_g1_fixed_ctnu",
        "role": "primary", "expected_features": 306, "expected_celltypes": 8,
        "expected_dropped_cells": 25,
        "original_run": BASE / "normalized_state_bhm_v2p2_celltype_nu" / (
            "v2p2_g1_fixed_ctnu__immune_coarse_primary__within_component_logcpm_z_cap5"
            "__all__linear__theta_0__publication"),
        "original_table": "immune_coarse_v2p2_feature_bmi_posterior.csv",
    },
    "nonimmune": {
        "compartment": "non_immune", "tier_column": "strict_qc_keep",
        "nu_scope": "global", "architecture": "v2p1c_g1_fixed_gnu",
        "role": "primary", "expected_features": 112, "expected_celltypes": 4,
        "expected_dropped_cells": 2,
        "original_run": BASE / "normalized_state_bhm_v2p1c_separate_compartments" / (
            "v2p1c_g1_fixed_gnu__nonimmune_primary__within_component_logcpm_z_cap5"
            "__all__linear__theta_0__publication"),
        "original_table": "nonimmune_v2p1c_feature_bmi_posterior.csv",
    },
    "immune_fine": {
        "compartment": "immune_fine", "tier_column": "tiered_primary_keep",
        "nu_scope": "global", "architecture": "v2p1c_g1_fixed_gnu",
        "role": "secondary", "expected_features": 496, "expected_celltypes": 14,
        "expected_dropped_cells": 58,
        "original_run": BASE / "normalized_state_bhm_v2p1c_separate_compartments" / (
            "v2p1c_g1_fixed_gnu__immune_fine_secondary__within_component_logcpm_z_cap5"
            "__all__linear__theta_0__publication"),
        "original_table": "immune_fine_v2p1c_feature_bmi_posterior.csv",
    },
}

PROFILES = {
    "smoke": {"tune": 500, "draws": 500, "chains": 2, "target_accept": 0.97},
    "standard": {"tune": 1000, "draws": 1000, "chains": 4, "target_accept": 0.98},
    "publication": {"tune": 2000, "draws": 4000, "chains": 4, "target_accept": 0.99},
}
PREDECESSOR = {"smoke": None, "standard": "smoke", "publication": "standard"}
PPC_DRAWS = {"smoke": 200, "standard": 2000, "publication": 4000}

BASELINE_SD = 0.50
CELLTYPE_BMI_SD = 0.20
FEATURE_BMI_SD = 0.30
CELLTYPE_COVARIATE_SD = 0.25
FEATURE_COVARIATE_SD = 0.25
PATIENT_GLOBAL_SD = 0.50
LOG_SIGMA_LOCATION = math.log(0.65)
LOG_SIGMA_LOCATION_SD = 0.40
CELLTYPE_LOG_SIGMA_SD = 0.25
FEATURE_LOG_SIGMA_SD = 0.50
NU_MINUS_TWO_RATE = 0.10

PRIMARY_ROPE = 0.10
ROPE_GRID = (0.05, 0.10, 0.15, 0.20)
Q_LEVEL = 0.05
MAX_TREE_DEPTH = int(CONFIG.get("max_tree_depth", 10))
MAX_TREE_DEPTH_FRACTION = 0.01
BFMI_MIN = 0.30

SMOKE_RHAT_HARD_MAX = 1.10
GATES = {
    "smoke": {"rhat_max": 1.05, "ess_min": 100},
    "standard": {"rhat_max": 1.01, "ess_min": 400},
    "publication": {"rhat_max": 1.01, "ess_min": 400},
}
PPC_ALPHA = 0.025
PPC_CORRELATION_THRESHOLD = 0.90
PPC_BATCH = 32

LOCKED_HASHES = {
    "pca_manifest.json": "a4263b46efb1c0d1d98a01b127ddb143bb7616ce88f8b364959613bbb74dcd30",
    "component_inclusion_tiers.csv": "16494a18ed467223134f0aed2efbf63c084d2f6b09332ca9854a9c403dd67294",
    "pca_input_hashes.csv": "6e2eaedbb5efe7c00d82e82c939d4416a5a3853f14bde881a55471cf874259c9",
    "combined_review_manifest.json": "04b0b1d788fc69e78ded4b44e050d487bfe231dfd5e5bd2b34f6b359c74f4476",
}
DECISION_RULE = {
    "any_bfdr_selection_change": True,
    "sign_flip_among_hdi_excluding_zero_features": True,
    "hdi_status_change_fraction_max": 0.10,
    "hdi_status_change_count_min": 3,
    "celltype_mean_slope_shift_max": 0.05,
}

if FAMILY not in FAMILY_SPECS:
    raise ValueError(f"FAMILY must be one of {sorted(FAMILY_SPECS)}")
if PROFILE not in PROFILES:
    raise ValueError(f"PROFILE must be one of {sorted(PROFILES)}")
if STAGE not in {"preflight", "fit", "diagnostics", "ppc", "compare"}:
    raise ValueError("Unknown STAGE")
if STAGE == "compare" and PROFILE != "publication":
    raise ValueError("compare uses the publication fit; set PROFILE = 'publication'")
if MIN_READS != 1000:
    warnings.warn("MIN_READS differs from the prespecified 1,000-read threshold.")

SPEC = FAMILY_SPECS[FAMILY]
PRIMARY_RUN_NAME = f"{SPEC['architecture']}__{FAMILY}__lowread{MIN_READS}"
RUN_NAME = f"{PRIMARY_RUN_NAME}__ri_hc" + ("" if MAX_TREE_DEPTH == 10 else f"_depth{MAX_TREE_DEPTH}")
RUN_SEED = 20260916 + int(hashlib.sha256(RUN_NAME.encode()).hexdigest()[:8], 16) % 1_000_000
ORIGINAL_RUN = Path(SPEC["original_run"])
if LOCAL_PATHS:
    PCA_DIR = Path(LOCAL_PATHS["pca_dir"])
    SENS_ROOT = Path(LOCAL_PATHS["sens_root"])
    ORIGINAL_RUN = Path(LOCAL_PATHS["original_runs"][FAMILY])
RUN_ROOT = SENS_ROOT / RUN_NAME
OUTPUT_DIR = RUN_ROOT / PROFILE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRACE_PATH = OUTPUT_DIR / "posterior_trace.nc"
print(f"{RUN_NAME} | profile {PROFILE} | stage {STAGE} | seed {RUN_SEED}")
print("Output:", OUTPUT_DIR)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

def write_json(value, path):
    tmp = Path(path).with_suffix(".tmp")
    tmp.write_text(json.dumps(value, indent=2, sort_keys=True, default=str))
    tmp.replace(path)

def write_csv(frame, path):
    tmp = Path(str(path) + ".tmp")
    frame.to_csv(tmp, index=False)
    tmp.replace(path)

def _clean_attrs(attrs):
    for key in list(attrs):
        value = attrs[key]
        if value is None or isinstance(value, (dict, set, bool)) or (
                isinstance(value, (list, tuple)) and any(isinstance(v, (dict, list, tuple, type(None))) for v in value)):
            attrs[key] = json.dumps(value, default=str, sort_keys=True)

def sanitize_attrs(idata):
    """netCDF rejects nested metadata (nutpie stores a dict); store it as JSON text."""
    _clean_attrs(idata.attrs)
    for group in idata.groups():
        _clean_attrs(idata[group].attrs)

def strict_bool(series):
    mapped = series.map({True: True, False: False, "True": True, "False": False,
                         "true": True, "false": False})
    if mapped.isna().any():
        raise ValueError("Non-boolean inclusion flag")
    return mapped.astype(bool)

# 4. Load the frozen inputs and apply the low-read rule
for name, expected in LOCKED_HASHES.items():
    folder = PCA_DIR
    if name == "combined_review_manifest.json":
        continue
    actual = sha256_file(folder / name)
    if actual != expected:
        raise RuntimeError(f"{name} does not match the locked hash")

pca_manifest = json.loads((PCA_DIR / "pca_manifest.json").read_text())
REVIEW_DIR = Path(pca_manifest["combined_review_directory"])
if LOCAL_PATHS:
    REVIEW_DIR = Path(LOCAL_PATHS["review_dir"])
review_manifest_path = REVIEW_DIR / "combined_review_manifest.json"
if sha256_file(review_manifest_path) != LOCKED_HASHES["combined_review_manifest.json"]:
    raise RuntimeError("combined_review_manifest.json does not match the locked hash")
review_manifest = json.loads(review_manifest_path.read_text())
RUN_DIR = Path(review_manifest["selected_runs"][SPEC["compartment"]])
if LOCAL_PATHS:
    RUN_DIR = Path(LOCAL_PATHS["run_dirs"][SPEC["compartment"]])

input_hashes = pd.read_csv(PCA_DIR / "pca_input_hashes.csv")
expected_input = input_hashes[input_hashes["compartment"].eq(SPEC["compartment"])]
cohort_path = RUN_DIR / "analysis_cohort.csv"
score_path = RUN_DIR / "audit_scores_all_variants.csv.gz"
qc_path = RUN_DIR / "patient_qc.csv"
for kind, path in (("cohort", cohort_path), ("scores", score_path)):
    locked = expected_input.loc[expected_input["input"].eq(kind), "sha256"]
    if len(locked) != 1 or sha256_file(path) != locked.iloc[0]:
        raise RuntimeError(f"{path.name} does not match pca_input_hashes.csv")

cohort = pd.read_csv(cohort_path, dtype={"base_sample_id": str})
scores = pd.read_csv(score_path, dtype={"base_sample_id": str})
patient_qc = pd.read_csv(qc_path, dtype={"base_sample_id": str})
tiers = pd.read_csv(PCA_DIR / "component_inclusion_tiers.csv")

if cohort["base_sample_id"].nunique() != 127:
    raise ValueError("Expected 127 patients")
if cohort["bmi_group"].value_counts().to_dict() != {"overweight": 57, "normal_weight": 52, "obese": 18}:
    raise ValueError("BMI group counts changed")

local_tiers = tiers[tiers["compartment"].eq(SPEC["compartment"])]
keep_celltypes = sorted(local_tiers.loc[strict_bool(local_tiers[SPEC["tier_column"]]), "CellType"].astype(str))
covariate_columns = ["base_sample_id", "bmi", "bmi_z", "bmi_group", "age_z", "sex_centered"]
frame = scores[scores["variant"].eq(SCORE_VARIANT) & scores["CellType"].isin(keep_celltypes)].copy()
frame["global_celltype"] = SPEC["compartment"] + "||" + frame["CellType"].astype(str)
frame["global_feature"] = SPEC["compartment"] + "||" + frame["feature"].astype(str)
frame = frame.merge(cohort[covariate_columns], on="base_sample_id", how="inner", validate="many_to_one")
if frame.duplicated(["base_sample_id", "global_feature"]).any():
    raise ValueError("Duplicate patient/feature rows")

frame["score"] = pd.to_numeric(frame["score"], errors="coerce")
grouped = frame.groupby("global_feature")["score"]
frame["score_model"] = (frame["score"] - grouped.transform("mean")) / grouped.transform("std")
frame["bmi_standardized"] = frame["bmi_z"]
if not np.isfinite(frame[["score_model", "bmi_standardized", "age_z", "sex_centered"]].to_numpy(float)).all():
    raise ValueError("Nonfinite model values")
if not frame.groupby("global_feature")["base_sample_id"].nunique().eq(127).all():
    raise ValueError("Every feature must contain all 127 patients before the low-read rule")

n_features = frame["global_feature"].nunique()
n_celltypes = frame["global_celltype"].nunique()
if (n_features, n_celltypes) != (SPEC["expected_features"], SPEC["expected_celltypes"]):
    raise ValueError(f"Expected {SPEC['expected_features']} features / {SPEC['expected_celltypes']} "
                     f"cell types, found {n_features} / {n_celltypes}")

original_inventory_path = ORIGINAL_RUN / "feature_inventory.csv"
if original_inventory_path.exists():
    original_features = set(pd.read_csv(original_inventory_path)["global_feature"])
    if original_features != set(frame["global_feature"]):
        raise ValueError("Feature set differs from the final fit's feature_inventory.csv")
    print("Feature set matches the final fit.")
else:
    warnings.warn(f"Cannot check against {original_inventory_path}")

reads = patient_qc[["base_sample_id", "CellType", "component_total", "theta"]].copy()
reads["component_total"] = pd.to_numeric(reads["component_total"], errors="coerce")
frame = frame.merge(reads, on=["base_sample_id", "CellType"], how="left", validate="many_to_one")
if frame["component_total"].isna().any():
    raise ValueError("Missing allocated-read totals")
frame["low_read"] = frame["component_total"] < MIN_READS

dropped_cells = (frame.loc[frame["low_read"], ["base_sample_id", "CellType", "component_total", "theta", "bmi"]]
                 .drop_duplicates(["base_sample_id", "CellType"])
                 .sort_values(["CellType", "component_total"]).reset_index(drop=True))
n_dropped_obs = int(frame["low_read"].sum())
if len(dropped_cells) != SPEC["expected_dropped_cells"] and MIN_READS == 1000:
    raise ValueError(f"Expected {SPEC['expected_dropped_cells']} low-read cells, found {len(dropped_cells)}")

full_frame = frame.copy()
analysis_data = frame.loc[~frame["low_read"]].reset_index(drop=True)
per_feature = analysis_data.groupby("global_feature")["base_sample_id"].nunique()
if per_feature.min() < 30:
    raise ValueError(f"A feature keeps only {per_feature.min()} patients")
if analysis_data["base_sample_id"].nunique() != 127:
    raise ValueError("A patient lost every observation")

write_csv(dropped_cells, RUN_ROOT / "dropped_low_read_cells.csv")
preflight = {
    "notebook_version": NOTEBOOK_VERSION, "family": FAMILY, "role": SPEC["role"],
    "architecture": SPEC["architecture"], "nu_scope": SPEC["nu_scope"],
    "min_reads": MIN_READS, "seed": RUN_SEED,
    "n_patients": 127, "n_features": n_features, "n_celltypes": n_celltypes,
    "n_observations_full": len(full_frame), "n_observations_fitted": len(analysis_data),
    "n_dropped_cells": len(dropped_cells), "n_dropped_observations": n_dropped_obs,
    "dropped_cells_by_celltype": dropped_cells["CellType"].value_counts().to_dict(),
    "minimum_patients_per_feature": int(per_feature.min()),
    "standardisation": "127-patient feature mean/SD from the final fits (before removal)",
    "input_sha256": {"cohort": sha256_file(cohort_path), "scores": sha256_file(score_path),
                     "patient_qc": sha256_file(qc_path)},
    "decision_rule": DECISION_RULE,
}
print(json.dumps(preflight, indent=2, default=str))
display_cells = dropped_cells.copy()
display_cells["component_total"] = display_cells["component_total"].round(1)
print(display_cells.to_string(index=False))

# 5. Build the model (final architecture, incomplete grid)
import pymc as pm
import pytensor.tensor as pt
import arviz as az

def prepare_arrays(data):
    data = data.sort_values(["global_celltype", "global_feature", "base_sample_id"]).reset_index(drop=True).copy()
    features = sorted(data["global_feature"].unique())
    celltypes = sorted(data["global_celltype"].unique())
    patients = sorted(full_frame["base_sample_id"].unique())
    f_index = {n: i for i, n in enumerate(features)}
    c_index = {n: i for i, n in enumerate(celltypes)}
    p_index = {n: i for i, n in enumerate(patients)}
    data["feature_idx"] = data["global_feature"].map(f_index)
    data["celltype_idx"] = data["global_celltype"].map(c_index)
    data["patient_idx"] = data["base_sample_id"].map(p_index)
    mapping = data[["feature_idx", "celltype_idx"]].drop_duplicates().sort_values("feature_idx")
    if len(mapping) != len(features):
        raise ValueError("Every feature must map to one cell type")
    feature_to_celltype = mapping["celltype_idx"].to_numpy(int)

    predictors = ["bmi_standardized", "age_z", "sex_centered"]
    table = (full_frame[["base_sample_id", *predictors]].drop_duplicates("base_sample_id")
             .set_index("base_sample_id").loc[patients])
    design = np.column_stack([np.ones(len(patients)), table.to_numpy(float)])
    if np.linalg.matrix_rank(design) != design.shape[1]:
        raise ValueError("Patient design is rank deficient")
    u, s, _ = np.linalg.svd(design, full_matrices=True)
    rank = int((s > max(design.shape) * np.finfo(float).eps * s.max()).sum())
    basis = u[:, rank:] * np.sqrt(len(patients) / (len(patients) - rank))
    if np.abs(design.T @ basis).max() > 1e-10:
        raise AssertionError("Patient basis is not orthogonal")
    return {"data": data, "features": features, "celltypes": celltypes, "patients": patients,
            "feature_to_celltype": feature_to_celltype, "patient_basis": basis,
            "design_rank": rank,
            "patient_covariates": table.to_numpy(float)}

def build_model(arrays):
    data = arrays["data"]
    covariates = ["age_z", "sex_centered"]
    coords = {"obs_id": np.arange(len(data)), "feature": arrays["features"],
              "celltype": arrays["celltypes"], "patient": arrays["patients"],
              "patient_re_dimension": np.arange(arrays["patient_basis"].shape[1]),
              "covariate": covariates}
    with pm.Model(coords=coords) as model:
        feature_idx = pm.Data("feature_idx", data["feature_idx"].to_numpy(int), dims="obs_id")
        patient_idx = pm.Data("patient_idx", data["patient_idx"].to_numpy(int), dims="obs_id")
        celltype_idx = pm.Data("celltype_idx", data["celltype_idx"].to_numpy(int), dims="obs_id")
        f2c = pm.Data("feature_to_celltype", arrays["feature_to_celltype"], dims="feature")
        bmi_z = pm.Data("bmi_z", data["bmi_standardized"].to_numpy(float), dims="obs_id")
        cov = pm.Data("covariate_matrix", data[covariates].to_numpy(float), dims=("obs_id", "covariate"))
        y = pm.Data("y", data["score_model"].to_numpy(float), dims="obs_id")
        patient_bmi = pm.Data("patient_bmi", arrays["patient_covariates"][:, 0], dims="patient")
        patient_cov = pm.Data("patient_covariate_matrix", arrays["patient_covariates"][:, 1:],
                              dims=("patient", "covariate"))

        feature_baseline = pm.Normal("feature_baseline", 0.0, BASELINE_SD, dims="feature")
        celltype_bmi_slope = pm.Normal("celltype_bmi_slope", 0.0, CELLTYPE_BMI_SD, dims="celltype")
        feature_sigma_bmi = pm.HalfNormal("feature_sigma_bmi", FEATURE_BMI_SD, dims="celltype")
        feature_bmi_raw = pm.Normal("feature_bmi_raw", 0.0, 1.0, dims="feature")
        feature_bmi_slope = pm.Deterministic(
            "feature_bmi_slope",
            celltype_bmi_slope[f2c] + feature_bmi_raw * feature_sigma_bmi[f2c], dims="feature")

        celltype_covariate_slope = pm.Normal("celltype_covariate_slope", 0.0, CELLTYPE_COVARIATE_SD,
                                             dims=("covariate", "celltype"))
        feature_sigma_covariate = pm.HalfNormal("feature_sigma_covariate", FEATURE_COVARIATE_SD,
                                                dims=("covariate", "celltype"))
        feature_covariate_raw = pm.Normal("feature_covariate_raw", 0.0, 1.0, dims=("covariate", "feature"))
        feature_covariate_slope = pm.Deterministic(
            "feature_covariate_slope",
            celltype_covariate_slope[:, f2c] + feature_covariate_raw * feature_sigma_covariate[:, f2c],
            dims=("covariate", "feature"))

        celltype_intercept_sigma = pm.HalfNormal("celltype_intercept_sigma", CELLTYPE_INTERCEPT_SD,
                                                 dims="celltype")
        w_mean = (patient_bmi[:, None] * celltype_bmi_slope[None, :]
                  + pt.dot(patient_cov, celltype_covariate_slope))
        celltype_intercept_w = pm.Normal("celltype_intercept_w", w_mean, celltype_intercept_sigma[None, :],
                                         dims=("patient", "celltype"))
        bmi_deviation = feature_bmi_raw * feature_sigma_bmi[f2c]
        covariate_deviation = feature_covariate_raw * feature_sigma_covariate[:, f2c]
        mu = (feature_baseline[feature_idx] + bmi_deviation[feature_idx] * bmi_z
              + celltype_intercept_w[patient_idx, celltype_idx]
              + pt.sum(cov * covariate_deviation[:, feature_idx].T, axis=1))

        log_sigma_location = pm.Normal("log_sigma_location", LOG_SIGMA_LOCATION, LOG_SIGMA_LOCATION_SD)
        log_sigma_celltype_location_sd = pm.HalfNormal("log_sigma_celltype_location_sd", CELLTYPE_LOG_SIGMA_SD)
        log_sigma_celltype_raw = pm.Normal("log_sigma_celltype_raw", 0.0, 1.0, dims="celltype")
        log_sigma_celltype_location = pm.Deterministic(
            "log_sigma_celltype_location",
            log_sigma_location + log_sigma_celltype_location_sd * log_sigma_celltype_raw, dims="celltype")
        log_sigma_feature_scale = pm.HalfNormal("log_sigma_feature_scale", FEATURE_LOG_SIGMA_SD, dims="celltype")
        log_sigma_raw = pm.Normal("log_sigma_raw", 0.0, 1.0, dims="feature")
        sigma_feature = pm.Deterministic(
            "sigma_feature",
            pt.exp(log_sigma_celltype_location[f2c] + log_sigma_feature_scale[f2c] * log_sigma_raw),
            dims="feature")

        if SPEC["nu_scope"] == "celltype":
            nu_minus_two = pm.Exponential("nu_minus_two", lam=NU_MINUS_TWO_RATE, dims="celltype")
            nu = pm.Deterministic("nu", nu_minus_two + 2.0, dims="celltype")
            nu_obs = nu[celltype_idx]
        else:
            nu_minus_two = pm.Exponential("nu_minus_two", lam=NU_MINUS_TWO_RATE)
            nu = pm.Deterministic("nu", nu_minus_two + 2.0)
            nu_obs = nu
        pm.StudentT("y_obs", nu=nu_obs, mu=mu, sigma=sigma_feature[feature_idx], observed=y, dims="obs_id")
    return model

arrays = prepare_arrays(analysis_data)
model = build_model(arrays)
initial_logp = float(model.compile_logp()(model.initial_point()))
if not np.isfinite(initial_logp):
    raise ValueError("Initial log density is not finite")
expected_rvs = {"feature_baseline", "celltype_bmi_slope", "feature_sigma_bmi", "feature_bmi_raw",
                "celltype_covariate_slope", "feature_sigma_covariate", "feature_covariate_raw",
                "celltype_intercept_sigma", "celltype_intercept_w", "log_sigma_location",
                "log_sigma_celltype_location_sd", "log_sigma_celltype_raw",
                "log_sigma_feature_scale", "log_sigma_raw", "nu_minus_two"}
if {rv.name for rv in model.free_RVs} != expected_rvs:
    raise AssertionError("Free random variables differ from the ri_hc architecture")
if (model["nu"].ndim == 1) != (SPEC["nu_scope"] == "celltype"):
    raise AssertionError("nu scope does not match the final architecture")

structure = {**preflight, "initial_logp": initial_logp,
             "patient_basis_dimensions": int(arrays["patient_basis"].shape[1]),
             "free_rvs": sorted(expected_rvs)}
write_json(structure, RUN_ROOT / "preflight_manifest.json")
print(f"Model built: {len(arrays['data']):,} observations, initial logp {initial_logp:.1f}")

# 6. Fit (with enforced promotion) and convergence gates
def gate_path(profile):
    return RUN_ROOT / profile / "run_diagnostics.json"

def compute_diagnostics(idata, profile):
    post = idata.posterior
    names = [v for v in post.data_vars]
    rhat = az.rhat(post[names])
    ess_bulk = az.ess(post[names], method="bulk")
    ess_tail = az.ess(post[names], method="tail")
    rows = []
    for name in names:
        r = np.atleast_1d(rhat[name].values).ravel()
        b = np.atleast_1d(ess_bulk[name].values).ravel()
        t = np.atleast_1d(ess_tail[name].values).ravel()
        rows.append({"parameter_family": name, "n": r.size, "max_rhat": float(np.nanmax(r)),
                     "min_ess_bulk": float(np.nanmin(b)), "min_ess_tail": float(np.nanmin(t)),
                     "nonfinite": int((~np.isfinite(r)).sum() + (~np.isfinite(b)).sum())})
    families = pd.DataFrame(rows).sort_values("max_rhat", ascending=False)
    stats = idata.sample_stats
    divergences = int(stats["diverging"].values.sum())
    bfmi = [float(x) for x in np.atleast_1d(az.bfmi(idata))]
    if "maxdepth_reached" in stats:
        maxdepth_fraction = stats["maxdepth_reached"].values.reshape(len(bfmi), -1).mean(axis=1)
    else:
        maxdepth_fraction = (stats["tree_depth"].values >= MAX_TREE_DEPTH).reshape(len(bfmi), -1).mean(axis=1)
    gate = GATES[profile]
    record = {
        "family": FAMILY, "profile": profile, "run_name": RUN_NAME,
        "chains": int(post.sizes["chain"]), "draws_per_chain": int(post.sizes["draw"]),
        "divergences": divergences, "min_bfmi": min(bfmi),
        "max_maxdepth_fraction": float(np.max(maxdepth_fraction)),
        "max_rhat": float(families["max_rhat"].max()),
        "min_ess_bulk": float(families["min_ess_bulk"].min()),
        "min_ess_tail": float(families["min_ess_tail"].min()),
        "nonfinite_diagnostics": int(families["nonfinite"].sum()),
        "trace_sha256": sha256_file(TRACE_PATH) if TRACE_PATH.exists() else None,
        "code_test_override": SAMPLER_OVERRIDE,
        "promotion_waiver": (json.loads((OUTPUT_DIR / "promotion_waiver.json").read_text())["reason"]
                             if (OUTPUT_DIR / "promotion_waiver.json").exists() else None),
    }
    record["gate_pass"] = bool(
        SAMPLER_OVERRIDE is None and divergences == 0 and min(bfmi) >= BFMI_MIN
        and record["max_maxdepth_fraction"] <= MAX_TREE_DEPTH_FRACTION
        and record["nonfinite_diagnostics"] == 0
        and record["max_rhat"] < gate["rhat_max"]
        and record["min_ess_bulk"] >= gate["ess_min"] and record["min_ess_tail"] >= gate["ess_min"])
    return record, families

idata = None
if STAGE == "fit":
    predecessor = PREDECESSOR[PROFILE]
    if predecessor is not None and SAMPLER_OVERRIDE is None:
        path = gate_path(predecessor)
        passed = path.exists() and json.loads(path.read_text()).get("gate_pass")
        if not passed:
            if not PROMOTION_WAIVER:
                raise RuntimeError(f"{PROFILE} requires a passed {predecessor} gate ({path}); "
                                   "see PROMOTION_WAIVER in cell 3")
            smoke_file = gate_path("smoke")
            if not smoke_file.exists():
                raise RuntimeError("A waiver needs a completed smoke fit")
            smoke = json.loads(smoke_file.read_text())
            geometry_ok = (smoke["divergences"] == 0 and smoke["min_bfmi"] >= BFMI_MIN
                           and smoke["max_maxdepth_fraction"] <= MAX_TREE_DEPTH_FRACTION
                           and smoke["nonfinite_diagnostics"] == 0 and smoke["max_rhat"] < SMOKE_RHAT_HARD_MAX
                           and smoke["code_test_override"] is None)
            if not geometry_ok:
                raise RuntimeError("Waiver refused: the smoke fit has a sampler-geometry problem")
            write_json({"profile": PROFILE, "failed_or_missing_predecessor": predecessor,
                        "predecessor_record": json.loads(path.read_text()) if path.exists() else None,
                        "smoke_record": smoke, "reason": PROMOTION_WAIVER,
                        "binding_gate": "publication: R-hat < 1.01, bulk/tail ESS >= 400, 0 divergences"},
                       OUTPUT_DIR / "promotion_waiver.json")
            print("Promotion waiver recorded:", PROMOTION_WAIVER)
    if TRACE_PATH.exists():
        raise FileExistsError(f"{TRACE_PATH} exists; never overwrite a trace")
    sampler = dict(PROFILES[PROFILE])
    if SAMPLER_OVERRIDE:
        sampler.update(SAMPLER_OVERRIDE)
        warnings.warn("SAMPLER_OVERRIDE is set: this run is a code test and can never pass a gate")
    start = time.time()
    with model:
        idata = pm.sample(tune=sampler["tune"], draws=sampler["draws"], chains=sampler["chains"],
                          cores=sampler["chains"], target_accept=sampler["target_accept"],
                          nuts_sampler="nutpie", nuts_sampler_kwargs={"maxdepth": MAX_TREE_DEPTH},
                          random_seed=RUN_SEED, return_inferencedata=True, progressbar=False)
    elapsed = time.time() - start
    idata.attrs.update({"run_name": RUN_NAME, "profile": PROFILE, "notebook_version": NOTEBOOK_VERSION})
    sanitize_attrs(idata)
    tmp = TRACE_PATH.with_name(".tmp_" + TRACE_PATH.name)
    idata.to_netcdf(tmp)
    tmp.replace(TRACE_PATH)
    print(f"Sampling took {elapsed / 60:.1f} min")
elif STAGE in {"diagnostics", "ppc", "compare"}:
    trace_profile = "publication" if STAGE == "compare" else PROFILE
    path = RUN_ROOT / trace_profile / "posterior_trace.nc"
    if not path.exists():
        raise FileNotFoundError(path)
    idata = az.from_netcdf(path)

if STAGE in {"fit", "diagnostics"}:
    diagnostics, family_table = compute_diagnostics(idata, PROFILE)
    write_json(diagnostics, OUTPUT_DIR / "run_diagnostics.json")
    write_csv(family_table, OUTPUT_DIR / "diagnostic_families.csv")
    print(json.dumps(diagnostics, indent=2))
    print(family_table.head(10).to_string(index=False))

# 7. Draw-matched posterior predictive check (incomplete grid)
def nan_residual_stats(matrix):
    """matrix: draws x features x patients with NaN for removed cells."""
    sd = np.nanstd(matrix, axis=2, ddof=1)
    centred = matrix - np.nanmean(matrix, axis=2, keepdims=True)
    tail = np.nanmax(np.abs(centred), axis=2) / sd
    return sd, tail

def dependence_count(matrix, columns):
    x = matrix[:, columns, :]
    if x.shape[1] < 2:
        return np.zeros(x.shape[0], dtype=np.int32)
    x = x - np.nanmean(x, axis=2, keepdims=True)
    x = np.nan_to_num(x, nan=0.0)
    norms = np.sqrt((x * x).sum(axis=2))
    corr = np.einsum("bip,bjp->bij", x, x, optimize=True) / (norms[:, :, None] * norms[:, None, :])
    upper = np.triu_indices(x.shape[1], k=1)
    return (np.abs(corr[:, upper[0], upper[1]]) >= PPC_CORRELATION_THRESHOLD).sum(axis=1).astype(np.int32)

def upper_p(reference, observed):
    return float((1 + np.sum(reference >= observed)) / (reference.size + 1))

def lower_p(reference, observed):
    return float((1 + np.sum(reference <= observed)) / (reference.size + 1))

if STAGE == "ppc":
    diag_file = OUTPUT_DIR / "run_diagnostics.json"
    if not diag_file.exists():
        raise RuntimeError("Run the diagnostics for this profile first")
    diag_record = json.loads(diag_file.read_text())
    if not diag_record["gate_pass"] and SAMPLER_OVERRIDE is None:
        raise RuntimeError("PPC requires a trace that passed its convergence gate")

    data = arrays["data"]
    features, patients = arrays["features"], arrays["patients"]
    n_f, n_p, n_c = len(features), len(patients), len(arrays["celltypes"])
    f_idx = data["feature_idx"].to_numpy(int)
    p_idx = data["patient_idx"].to_numpy(int)
    c_idx = data["celltype_idx"].to_numpy(int)
    y_obs = data["score_model"].to_numpy(float)
    bmi = data["bmi_standardized"].to_numpy(float)
    covariates = data[["age_z", "sex_centered"]].to_numpy(float)
    grid = np.full((n_f, n_p), -1, dtype=np.int64)
    grid[f_idx, p_idx] = np.arange(len(data))
    observed_mask = grid >= 0
    columns_by_celltype = [np.flatnonzero(arrays["feature_to_celltype"] == j) for j in range(n_c)]

    def to_grid(values):
        out = np.full((values.shape[0], n_f, n_p), np.nan)
        out[:, observed_mask] = values[:, grid[observed_mask]]
        return out

    post = idata.posterior
    n_chains, n_draws = post.sizes["chain"], post.sizes["draw"]
    requested = PPC_DRAWS_OVERRIDE or PPC_DRAWS[PROFILE]
    requested = min(requested - requested % (2 * n_chains), 2 * n_chains * (n_draws // 2))
    per_chain = requested // n_chains
    positions = np.linspace(0, n_draws - 1, per_chain).round().astype(int)
    chain_of = np.repeat(np.arange(n_chains), per_chain)
    draw_of = np.tile(positions, n_chains)
    rng = np.random.default_rng(RUN_SEED + 70_000)

    raw_obs_grid = to_grid(y_obs[None, :])
    observed_raw_dep = np.array([dependence_count(raw_obs_grid, c)[0] for c in columns_by_celltype])

    stats = {k: [] for k in ("rep_sd", "rep_tail", "obs_sd", "obs_tail")}
    dep = {k: [] for k in ("rep_raw", "rep_res", "obs_res")}
    for start in range(0, requested, PPC_BATCH):
        sel = slice(start, min(start + PPC_BATCH, requested))
        import xarray as xr
        pick = post.isel(chain=xr.DataArray(chain_of[sel], dims="k"), draw=xr.DataArray(draw_of[sel], dims="k"))
        g = lambda name, dims: np.asarray(pick[name].transpose("k", *dims), float)
        base = g("feature_baseline", ("feature",))
        slope = g("feature_bmi_slope", ("feature",))
        cslope = g("feature_covariate_slope", ("covariate", "feature"))
        w = g("celltype_intercept_w", ("patient", "celltype"))
        ctb = g("celltype_bmi_slope", ("celltype",))
        ctc = g("celltype_covariate_slope", ("covariate", "celltype"))
        pcov = arrays["patient_covariates"]
        v = w - (pcov[None, :, 0, None] * ctb[:, None, :]
                 + np.einsum("pc,bck->bpk", pcov[:, 1:], ctc, optimize=True))
        sigma = g("sigma_feature", ("feature",))
        nu = g("nu", ("celltype",)) if SPEC["nu_scope"] == "celltype" else g("nu", ())[:, None]
        mu = (base[:, f_idx] + slope[:, f_idx] * bmi + v[:, p_idx, c_idx]
              + np.einsum("oc,bco->bo", covariates, cslope[:, :, f_idx], optimize=True))
        nu_o = nu[:, c_idx] if SPEC["nu_scope"] == "celltype" else np.broadcast_to(nu, mu.shape)
        y_rep = mu + sigma[:, f_idx] * rng.standard_t(nu_o)

        rep_raw_grid = to_grid(y_rep)
        rep_res_grid = to_grid(y_rep - mu)
        obs_res_grid = to_grid(y_obs[None, :] - mu)
        s1, t1 = nan_residual_stats(rep_res_grid)
        s2, t2 = nan_residual_stats(obs_res_grid)
        stats["rep_sd"].append(s1); stats["rep_tail"].append(t1)
        stats["obs_sd"].append(s2); stats["obs_tail"].append(t2)
        dep["rep_raw"].append(np.column_stack([dependence_count(rep_raw_grid, c) for c in columns_by_celltype]))
        dep["rep_res"].append(np.column_stack([dependence_count(rep_res_grid, c) for c in columns_by_celltype]))
        dep["obs_res"].append(np.column_stack([dependence_count(obs_res_grid, c) for c in columns_by_celltype]))
        del rep_raw_grid, rep_res_grid, obs_res_grid, y_rep, mu
        gc.collect()
    stats = {k: np.concatenate(v) for k, v in stats.items()}
    dep = {k: np.concatenate(v) for k, v in dep.items()}

    calibration = np.zeros(requested, bool)
    for chain in range(n_chains):
        idx = np.flatnonzero(chain_of == chain)
        calibration[idx[::2]] = True
    evaluation = ~calibration

    labels = ["__GLOBAL__", *arrays["celltypes"]]
    rows = []

    def add(domain, obs_ct, rep_ct):
        obs_all = np.column_stack([obs_ct.sum(axis=1), obs_ct])
        rep_all = np.column_stack([rep_ct.sum(axis=1), rep_ct])
        for j, label in enumerate(labels):
            o, r = obs_all[:, j], rep_all[:, j]
            up = (1 + np.sum(r >= o)) / (len(r) + 1)
            lo = (1 + np.sum(r <= o)) / (len(r) + 1)
            rows.append({"domain": domain, "celltype": label,
                         "observed_median": float(np.median(o)), "replicated_median": float(np.median(r)),
                         "replicated_q025": float(np.quantile(r, 0.025)),
                         "replicated_q975": float(np.quantile(r, 0.975)),
                         "upper_p": float(up), "lower_p": float(lo),
                         "flag_upper": bool(up <= PPC_ALPHA)})

    def ct_counts(flags):
        return np.column_stack([flags[:, c].sum(axis=1) for c in columns_by_celltype])

    for domain, obs_key, rep_key in (("residual_dispersion", "obs_sd", "rep_sd"),
                                     ("studentized_residual_tails", "obs_tail", "rep_tail")):
        low, high = np.quantile(stats[rep_key][calibration], [0.025, 0.975], axis=0)
        obs_flags = (stats[obs_key][evaluation] < low) | (stats[obs_key][evaluation] > high)
        rep_flags = (stats[rep_key][evaluation] < low) | (stats[rep_key][evaluation] > high)
        add(domain, ct_counts(obs_flags), ct_counts(rep_flags))
        if domain == "studentized_residual_tails":
            obs_tail_rate = obs_flags.mean(axis=0)
    add("raw_outcome_dependence",
        np.broadcast_to(observed_raw_dep, (evaluation.sum(), n_c)), dep["rep_raw"][evaluation])
    add("conditional_residual_dependence", dep["obs_res"][evaluation], dep["rep_res"][evaluation])

    checks = pd.DataFrame(rows)
    feature_checks = pd.DataFrame({
        "global_feature": features,
        "global_celltype": np.asarray(arrays["celltypes"])[arrays["feature_to_celltype"]],
        "n_patients_fitted": observed_mask.sum(axis=1),
        "observed_tail_median": np.median(stats["obs_tail"], axis=0),
        "replicated_tail_q975": np.quantile(stats["rep_tail"], 0.975, axis=0),
        "tail_outside_rate": obs_tail_rate,
    })
    glob_tail = checks[(checks.domain == "studentized_residual_tails") & (checks.celltype == "__GLOBAL__")].iloc[0]
    summary = {
        "ppc_version": "draw-matched-incomplete-grid-v1", "family": FAMILY, "profile": PROFILE,
        "joint_draws": int(requested), "evaluation_draws": int(evaluation.sum()),
        "trace_sha256": sha256_file(TRACE_PATH),
        "formal": PROFILE != "smoke" and SAMPLER_OVERRIDE is None,
        "tail_upper_p": float(glob_tail.upper_p),
        "tail_adequacy_pass": bool(glob_tail.upper_p > PPC_ALPHA),
        "global_domains_flagged": checks.loc[(checks.celltype == "__GLOBAL__") & checks.flag_upper, "domain"].tolist(),
        "dispersion_note": ("residual_dispersion uses feature-specific sigma fitted to the same data, "
                            "so it is weak by construction; read lower_p as well as upper_p"),
        "dependence_note": ("correlations centre each feature on its fitted patients and zero-fill removed "
                            "cells; the same rule is applied to observed and replicated data"),
    }
    write_csv(checks, OUTPUT_DIR / "ppc_domain_checks.csv")
    write_csv(feature_checks, OUTPUT_DIR / "ppc_feature_checks.csv")
    write_json(summary, OUTPUT_DIR / "ppc_summary.json")
    print(checks[checks.celltype == "__GLOBAL__"].to_string(index=False))
    print(json.dumps(summary, indent=2))

def summarise_slopes(post, name, dim):
    da = post[name].transpose(dim, "chain", "draw")
    ids = np.asarray(da.coords[dim], str)
    s = np.asarray(da).reshape(len(ids), -1)
    h = az.hdi(post[name], hdi_prob=0.95)[name].sel({dim: ids}).values
    return pd.DataFrame({dim: ids, "mean": s.mean(axis=1), "hdi_low": h[:, 0], "hdi_high": h[:, 1],
                         "p_positive": (s > 0).mean(axis=1)})

if STAGE == "compare":
    pub_diag = RUN_ROOT / "publication" / "run_diagnostics.json"
    pub_ppc = RUN_ROOT / "publication" / "ppc_summary.json"
    for path in (pub_diag, pub_ppc):
        if not path.exists():
            raise FileNotFoundError(f"compare needs {path}")
    if not json.loads(pub_diag.read_text())["gate_pass"] and SAMPLER_OVERRIDE is None:
        raise RuntimeError("The ri_hc publication fit did not pass its convergence gate")
    primary_root = PRIMARY_LOWREAD_ROOT / PRIMARY_RUN_NAME / "publication"
    prim_trace = primary_root / "posterior_trace.nc"
    if not prim_trace.exists():
        raise FileNotFoundError(f"Published low-read primary trace not found: {prim_trace}")
    prim = az.from_netcdf(prim_trace)
    bmi_sd = float(cohort["bmi"].std(ddof=1)); per5 = 5.0 / bmi_sd

    labels = full_frame[["global_feature", "CellType", "Signature"]].drop_duplicates("global_feature")
    new = summarise_slopes(idata.posterior, "feature_bmi_slope", "feature").add_prefix("new_")
    old = summarise_slopes(prim.posterior, "feature_bmi_slope", "feature").add_prefix("prim_")
    merged = (labels.merge(old.rename(columns={"prim_feature": "global_feature"}), on="global_feature")
                    .merge(new.rename(columns={"new_feature": "global_feature"}), on="global_feature"))
    if len(merged) != SPEC["expected_features"]:
        raise ValueError("Feature sets do not align with the primary fit")
    for c in [c for c in merged.columns if c.endswith(("mean", "hdi_low", "hdi_high"))]:
        merged[c] = merged[c] * per5
    merged["prim_wide_low"] = merged.prim_mean + WIDENING * (merged.prim_hdi_low - merged.prim_mean)
    merged["prim_wide_high"] = merged.prim_mean + WIDENING * (merged.prim_hdi_high - merged.prim_mean)
    excl = lambda lo, hi: (lo > 0) | (hi < 0)
    merged["prim_excl"] = excl(merged.prim_hdi_low, merged.prim_hdi_high)
    merged["prim_wide_excl"] = excl(merged.prim_wide_low, merged.prim_wide_high)
    merged["new_excl"] = excl(merged.new_hdi_low, merged.new_hdi_high)
    merged["width_ratio"] = (merged.new_hdi_high - merged.new_hdi_low) / (merged.prim_hdi_high - merged.prim_hdi_low)
    merged["sign_flip"] = np.sign(merged.new_mean) != np.sign(merged.prim_mean)

    ct_new = summarise_slopes(idata.posterior, "celltype_bmi_slope", "celltype").add_prefix("new_")
    ct_old = summarise_slopes(prim.posterior, "celltype_bmi_slope", "celltype").add_prefix("prim_")
    celltype = ct_old.rename(columns={"prim_celltype": "celltype"}).merge(
        ct_new.rename(columns={"new_celltype": "celltype"}), on="celltype")
    for c in [c for c in celltype.columns if c.endswith(("mean", "hdi_low", "hdi_high"))]:
        celltype[c] = celltype[c] * per5
    celltype["prim_excl"] = excl(celltype.prim_hdi_low, celltype.prim_hdi_high)
    celltype["new_excl"] = excl(celltype.new_hdi_low, celltype.new_hdi_high)
    celltype["width_ratio"] = (celltype.new_hdi_high - celltype.new_hdi_low) / (celltype.prim_hdi_high - celltype.prim_hdi_low)
    sig = idata.posterior["celltype_intercept_sigma"].mean(("chain", "draw")).to_pandas()
    celltype["intercept_sigma_mean"] = celltype["celltype"].map(sig)

    dep_cols = ["celltype", "observed_median", "replicated_q975", "upper_p", "flag_upper"]
    dep_new = pd.read_csv(RUN_ROOT / "publication" / "ppc_domain_checks.csv")
    dep_old_path = primary_root / "ppc_domain_checks.csv"
    dep_new = dep_new[dep_new.domain.eq("conditional_residual_dependence")][dep_cols]
    if dep_old_path.exists():
        dep_old = pd.read_csv(dep_old_path)
        dep_old = dep_old[dep_old.domain.eq("conditional_residual_dependence")][dep_cols]
        dependence = dep_old.merge(dep_new, on="celltype", suffixes=("_prim", "_new"))
    else:
        dependence = dep_new.add_suffix("_new").rename(columns={"celltype_new": "celltype"})

    wanted = pd.DataFrame(HEADLINE[FAMILY], columns=["CellType", "Signature"])
    headline = wanted.merge(merged, on=["CellType", "Signature"], how="left")
    missing = headline[headline.global_feature.isna()][["CellType", "Signature"]].values.tolist()

    agreement = pd.crosstab(merged.prim_wide_excl, merged.new_excl,
                            rownames=["primary_1.37x_excludes_0"], colnames=["ri_hc_excludes_0"])
    glob_flag = lambda d, col: bool(d.loc[d.celltype.eq("__GLOBAL__"), col].iloc[0]) if col in d else None
    dep_resolved = (glob_flag(dependence, "flag_upper_new") is False)
    decision = {
        "family": FAMILY, "model": "ri_replace_hc", "max_tree_depth": MAX_TREE_DEPTH,
        "units": "score SD per 5 kg/m2",
        "n_hdi_excluding_zero": {"primary": int(merged.prim_excl.sum()),
                                 "primary_widened_1.37": int(merged.prim_wide_excl.sum()),
                                 "ri_hc": int(merged.new_excl.sum()),
                                 "expected_if_all_null": round(0.05 * len(merged), 1)},
        "agreement_widened_vs_ri_hc": agreement.to_dict(),
        "sign_flips_among_excluding": int((merged.sign_flip & (merged.prim_excl | merged.new_excl)).sum()),
        "median_feature_width_ratio": float(merged.width_ratio.median()),
        "celltype_means_excluding_zero": {"primary": int(celltype.prim_excl.sum()), "ri_hc": int(celltype.new_excl.sum())},
        "dependence_global_flag": {"primary": glob_flag(dependence, "flag_upper_prim"),
                                   "ri_hc": glob_flag(dependence, "flag_upper_new")},
        "dependence_celltypes_flagged": {
            "primary": int(dependence.loc[dependence.celltype.ne("__GLOBAL__"), "flag_upper_prim"].sum()) if "flag_upper_prim" in dependence else None,
            "ri_hc": int(dependence.loc[dependence.celltype.ne("__GLOBAL__"), "flag_upper_new"].sum())},
        "headline_missing": missing,
        "ppc_summary": json.loads(pub_ppc.read_text()),
        "primary_trace_sha256": sha256_file(prim_trace),
        "ri_hc_trace_sha256": sha256_file(RUN_ROOT / "publication" / "posterior_trace.nc"),
    }
    decision["triggered"] = {"dependence_still_flagged": not dep_resolved}
    decision["decision"] = ("DEPENDENCE_RESOLVED" if dep_resolved else "DEPENDENCE_STILL_FLAGGED")
    compare_dir = RUN_ROOT / "comparison"
    compare_dir.mkdir(exist_ok=True)
    write_csv(merged, compare_dir / "feature_comparison.csv")
    write_csv(celltype, compare_dir / "celltype_comparison.csv")
    write_csv(dependence, compare_dir / "dependence_comparison.csv")
    write_csv(headline, compare_dir / "headline_comparison.csv")
    write_json(decision, compare_dir / "decision.json")
    show = ["CellType", "Signature", "prim_mean", "prim_hdi_low", "prim_hdi_high", "prim_wide_low",
            "new_mean", "new_hdi_low", "new_hdi_high", "width_ratio"]
    print(json.dumps({k: decision[k] for k in ("decision", "n_hdi_excluding_zero", "celltype_means_excluding_zero",
                                                "dependence_global_flag", "dependence_celltypes_flagged",
                                                "median_feature_width_ratio")}, indent=2))
    print(celltype.round(3).to_string(index=False))
    print(dependence.round(3).to_string(index=False))
    print(headline[show].round(3).to_string(index=False))


In [ ]:
%%writefile /content/sens_worker.py
import os, re, json, time
from pathlib import Path
import numpy as np, pandas as pd, arviz as az, pymc as pm, pytensor.tensor as pt

FAMILY = os.environ["SENS_FAMILY"]
JOB = os.environ["SENS_JOB"]
SEED = 20261119

# ---------------- jobs ----------------
JOBS = {
    "bmi_wide":            dict(kind="continuous", min_reads=1000, celltype_bmi_sd=0.40, feature_bmi_sd=0.60),
    "bmi_narrow":          dict(kind="continuous", min_reads=1000, celltype_bmi_sd=0.10, feature_bmi_sd=0.15),
    "int_wide":            dict(kind="continuous", min_reads=1000, intercept_sd=1.00),
    "int_narrow":          dict(kind="continuous", min_reads=1000, intercept_sd=0.25),
    "allprofiles":         dict(kind="continuous", min_reads=0),
    "categorical_all":     dict(kind="categorical", min_reads=0),
    "categorical_lowread": dict(kind="categorical", min_reads=1000),
}
J = JOBS[JOB]
CAT_CELLTYPE_SD, CAT_FEATURE_SD = 0.40, 0.50
SAMPLER = dict(tune=2000, draws=4000, chains=4, target_accept=0.99)

s = Path("/content/ri_hc_engine.py").read_text()
assert "ri-hc-t-" not in s, "cell 3 engine must be the unpatched engine"
edits = [
    ('NOTEBOOK_VERSION = "ri-hc-celltype-intercept-v1"',
     'NOTEBOOK_VERSION = "ri-hc-t-celltype-intercept-v1"'),
    ('RUN_NAME = f"{PRIMARY_RUN_NAME}__ri_hc"',
     'RUN_NAME = f"{PRIMARY_RUN_NAME}__ri_hc_t"'),
    ('''        celltype_intercept_w = pm.Normal("celltype_intercept_w", w_mean, celltype_intercept_sigma[None, :],
                                         dims=("patient", "celltype"))''',
     '''        intercept_nu_minus_two = pm.Exponential("intercept_nu_minus_two", lam=NU_MINUS_TWO_RATE)
        intercept_nu = pm.Deterministic("intercept_nu", intercept_nu_minus_two + 2.0)
        celltype_intercept_w = pm.StudentT("celltype_intercept_w", nu=intercept_nu, mu=w_mean,
                                           sigma=celltype_intercept_sigma[None, :],
                                           dims=("patient", "celltype"))'''),
    ('"log_sigma_feature_scale", "log_sigma_raw", "nu_minus_two"}',
     '"log_sigma_feature_scale", "log_sigma_raw", "nu_minus_two", "intercept_nu_minus_two"}'),
]
for old, new in edits:
    assert s.count(old) == 1, f"Student-t patch pattern not found exactly once:\n{old}"
    s = s.replace(old, new)

def setconst(name, value):
    global s
    s, n = re.subn(rf"^{name} = [0-9.]+", f"{name} = {value}", s, flags=re.M)
    assert n == 1, name

s = s.replace('__ri_hc_t"', f'__ri_hc_t_sens_{JOB}"')
assert f"__ri_hc_t_sens_{JOB}" in s
if "celltype_bmi_sd" in J: setconst("CELLTYPE_BMI_SD", J["celltype_bmi_sd"])
if "feature_bmi_sd" in J: setconst("FEATURE_BMI_SD", J["feature_bmi_sd"])
if "intercept_sd" in J: setconst("CELLTYPE_INTERCEPT_SD", J["intercept_sd"])

ns = {"CONFIG": {"family": FAMILY, "profile": "publication", "stage": "preflight", "waiver": None,
                 "max_tree_depth": 10, "min_reads": J["min_reads"]}, "__name__": "lowread_engine"}
exec(compile(s, "lowread_engine.py", "exec"), ns)
arrays = ns["arrays"]
data = arrays["data"]
OUT = Path(ns["SENS_ROOT"]) / "_primary_sensitivity" / f"{FAMILY}__{JOB}"
OUT.mkdir(parents=True, exist_ok=True)
if (OUT / "diagnostics.json").exists():
    print(f"{FAMILY}__{JOB}: already done", flush=True)
    raise SystemExit

coh = ns["cohort"]
per5 = 5.0 * float(np.polyfit(coh["bmi"].to_numpy(float), coh["bmi_z"].to_numpy(float), 1)[0])

def build_categorical(arrays):
    data = arrays["data"]
    covariates = ["age_z", "sex_centered"]
    contrasts = ["overweight_vs_normal", "obese_vs_normal"]
    xg = np.column_stack([data["bmi_group"].eq("overweight"), data["bmi_group"].eq("obese")]).astype(float)
    pmap = data[["patient_idx", "bmi_group"]].drop_duplicates().set_index("patient_idx")["bmi_group"]
    pg = pmap.reindex(range(len(arrays["patients"])))
    assert pg.notna().all()
    xp = np.column_stack([pg.eq("overweight"), pg.eq("obese")]).astype(float)
    coords = {"obs_id": np.arange(len(data)), "feature": arrays["features"], "celltype": arrays["celltypes"],
              "patient": arrays["patients"], "covariate": covariates, "contrast": contrasts}
    with pm.Model(coords=coords) as model:
        feature_idx = pm.Data("feature_idx", data["feature_idx"].to_numpy(int), dims="obs_id")
        patient_idx = pm.Data("patient_idx", data["patient_idx"].to_numpy(int), dims="obs_id")
        celltype_idx = pm.Data("celltype_idx", data["celltype_idx"].to_numpy(int), dims="obs_id")
        f2c = pm.Data("feature_to_celltype", arrays["feature_to_celltype"], dims="feature")
        x_obs = pm.Data("group_matrix", xg, dims=("obs_id", "contrast"))
        cov = pm.Data("covariate_matrix", data[covariates].to_numpy(float), dims=("obs_id", "covariate"))
        y = pm.Data("y", data["score_model"].to_numpy(float), dims="obs_id")
        patient_x = pm.Data("patient_group_matrix", xp, dims=("patient", "contrast"))
        patient_cov = pm.Data("patient_covariate_matrix", arrays["patient_covariates"][:, 1:], dims=("patient", "covariate"))

        feature_baseline = pm.Normal("feature_baseline", 0.0, ns["BASELINE_SD"], dims="feature")
        celltype_group_effect = pm.Normal("celltype_group_effect", 0.0, CAT_CELLTYPE_SD, dims=("contrast", "celltype"))
        feature_sigma_group = pm.HalfNormal("feature_sigma_group", CAT_FEATURE_SD, dims=("contrast", "celltype"))
        feature_group_raw = pm.Normal("feature_group_raw", 0.0, 1.0, dims=("contrast", "feature"))
        group_dev = feature_group_raw * feature_sigma_group[:, f2c]
        pm.Deterministic("feature_group_effect", celltype_group_effect[:, f2c] + group_dev, dims=("contrast", "feature"))
        pm.Deterministic("celltype_obese_vs_overweight", celltype_group_effect[1] - celltype_group_effect[0], dims="celltype")
        pm.Deterministic("feature_obese_vs_overweight",
                         (celltype_group_effect[1, f2c] + group_dev[1]) - (celltype_group_effect[0, f2c] + group_dev[0]),
                         dims="feature")

        celltype_covariate_slope = pm.Normal("celltype_covariate_slope", 0.0, ns["CELLTYPE_COVARIATE_SD"], dims=("covariate", "celltype"))
        feature_sigma_covariate = pm.HalfNormal("feature_sigma_covariate", ns["FEATURE_COVARIATE_SD"], dims=("covariate", "celltype"))
        feature_covariate_raw = pm.Normal("feature_covariate_raw", 0.0, 1.0, dims=("covariate", "feature"))
        cov_dev = feature_covariate_raw * feature_sigma_covariate[:, f2c]

        celltype_intercept_sigma = pm.HalfNormal("celltype_intercept_sigma", ns["CELLTYPE_INTERCEPT_SD"], dims="celltype")
        intercept_nu_minus_two = pm.Exponential("intercept_nu_minus_two", lam=ns["NU_MINUS_TWO_RATE"])
        intercept_nu = pm.Deterministic("intercept_nu", intercept_nu_minus_two + 2.0)
        w_mean = pt.dot(patient_x, celltype_group_effect) + pt.dot(patient_cov, celltype_covariate_slope)
        celltype_intercept_w = pm.StudentT("celltype_intercept_w", nu=intercept_nu, mu=w_mean,
                                           sigma=celltype_intercept_sigma[None, :], dims=("patient", "celltype"))
        mu = (feature_baseline[feature_idx] + pt.sum(x_obs * group_dev[:, feature_idx].T, axis=1)
              + celltype_intercept_w[patient_idx, celltype_idx] + pt.sum(cov * cov_dev[:, feature_idx].T, axis=1))

        log_sigma_location = pm.Normal("log_sigma_location", ns["LOG_SIGMA_LOCATION"], ns["LOG_SIGMA_LOCATION_SD"])
        log_sigma_celltype_location_sd = pm.HalfNormal("log_sigma_celltype_location_sd", ns["CELLTYPE_LOG_SIGMA_SD"])
        log_sigma_celltype_raw = pm.Normal("log_sigma_celltype_raw", 0.0, 1.0, dims="celltype")
        log_sigma_celltype_location = pm.Deterministic("log_sigma_celltype_location",
            log_sigma_location + log_sigma_celltype_location_sd * log_sigma_celltype_raw, dims="celltype")
        log_sigma_feature_scale = pm.HalfNormal("log_sigma_feature_scale", ns["FEATURE_LOG_SIGMA_SD"], dims="celltype")
        log_sigma_raw = pm.Normal("log_sigma_raw", 0.0, 1.0, dims="feature")
        sigma_feature = pm.Deterministic("sigma_feature",
            pt.exp(log_sigma_celltype_location[f2c] + log_sigma_feature_scale[f2c] * log_sigma_raw), dims="feature")
        if ns["SPEC"]["nu_scope"] == "celltype":
            nu_minus_two = pm.Exponential("nu_minus_two", lam=ns["NU_MINUS_TWO_RATE"], dims="celltype")
            nu = pm.Deterministic("nu", nu_minus_two + 2.0, dims="celltype")
            nu_obs = nu[celltype_idx]
        else:
            nu_minus_two = pm.Exponential("nu_minus_two", lam=ns["NU_MINUS_TWO_RATE"])
            nu = pm.Deterministic("nu", nu_minus_two + 2.0)
            nu_obs = nu
        pm.StudentT("y_obs", nu=nu_obs, mu=mu, sigma=sigma_feature[feature_idx], observed=y, dims="obs_id")
    return model

model = ns["model"] if J["kind"] == "continuous" else build_categorical(arrays)
print(f"{FAMILY}__{JOB}: model built ({len(data):,} observations, min_reads {J['min_reads']}, {J['kind']}); "
      f"sampling {SAMPLER}", flush=True)

t0 = time.time()
with model:
    idata = pm.sample(tune=SAMPLER["tune"], draws=SAMPLER["draws"], chains=SAMPLER["chains"], cores=SAMPLER["chains"],
                      target_accept=SAMPLER["target_accept"], nuts_sampler="nutpie",
                      nuts_sampler_kwargs={"maxdepth": 10}, random_seed=SEED, return_inferencedata=True, progressbar=False)
minutes = (time.time() - t0) / 60
ns["sanitize_attrs"](idata)
tmp = OUT / ".tmp_posterior_trace.nc"
idata.to_netcdf(tmp)
tmp.replace(OUT / "posterior_trace.nc")
ns["TRACE_PATH"] = OUT / "posterior_trace.nc"
record, families = ns["compute_diagnostics"](idata, "publication")
record.update({"family": FAMILY, "job": JOB, "job_settings": J, "minutes": minutes, "n_obs": int(len(data)),
               "per5_factor": per5})
families.to_csv(OUT / "diagnostic_families.csv", index=False)
post = idata.posterior

def summarise(da, dim, scale, label):
    da = da * scale
    h = az.hdi(da.to_dataset(name="v"), hdi_prob=0.95)["v"].transpose(dim, "hdi").values
    t = pd.DataFrame({dim: da[dim].values.astype(str), "estimand": label,
                      "mean": da.mean(("chain", "draw")).values, "hdi95_low": h[:, 0], "hdi95_high": h[:, 1],
                      "p_positive": (da > 0).mean(("chain", "draw")).values})
    t["excludes_zero"] = (t.hdi95_low > 0) | (t.hdi95_high < 0)
    return t

if J["kind"] == "continuous":
    ct = summarise(post["celltype_bmi_slope"], "celltype", per5, "per_5kg_m2")
    ft = summarise(post["feature_bmi_slope"], "feature", per5, "per_5kg_m2")
else:
    ct = pd.concat([summarise(post["celltype_group_effect"].sel(contrast=k), "celltype", 1.0, k)
                    for k in ["overweight_vs_normal", "obese_vs_normal"]]
                   + [summarise(post["celltype_obese_vs_overweight"], "celltype", 1.0, "obese_vs_overweight")])
    ft = pd.concat([summarise(post["feature_group_effect"].sel(contrast=k), "feature", 1.0, k)
                    for k in ["overweight_vs_normal", "obese_vs_normal"]]
                   + [summarise(post["feature_obese_vs_overweight"], "feature", 1.0, "obese_vs_overweight")])
ct.to_csv(OUT / "celltype_summary.csv", index=False)
ft.to_csv(OUT / "feature_summary.csv", index=False)
record["n_features_excluding_zero"] = {k: int(v) for k, v in ft.groupby("estimand")["excludes_zero"].sum().items()}
record["n_celltypes_excluding_zero"] = {k: int(v) for k, v in ct.groupby("estimand")["excludes_zero"].sum().items()}
(OUT / "diagnostics.json").write_text(json.dumps(record, indent=2, default=str))
print(f"{FAMILY}__{JOB}: gate_pass={record['gate_pass']} max_rhat={record['max_rhat']:.4f} "
      f"min_ess={record['min_ess_bulk']:.0f}/{record['min_ess_tail']:.0f} div={record['divergences']} "
      f"({minutes:.0f} min)", flush=True)


In [ ]:
import os, sys, subprocess, time
from pathlib import Path

RUN_JOBS = [('immune_coarse', 'bmi_wide'), ('immune_coarse', 'bmi_narrow'), ('immune_coarse', 'int_wide'), ('immune_coarse', 'int_narrow')]
SLOTS = max(1, (os.cpu_count() or 2) // 4)
ROOT = Path("/content/drive/MyDrive/new_dysfunction/normalized_state_bhm_ri_hc_v1/_primary_sensitivity")
LOGS = ROOT / "logs"; LOGS.mkdir(parents=True, exist_ok=True)
print(f"{os.cpu_count()} CPUs -> {SLOTS} jobs at a time; jobs here: {RUN_JOBS}")

def running():
    out = subprocess.run(["pgrep", "-fa", "sens_worker.py"], capture_output=True, text=True).stdout
    return [l for l in out.splitlines() if "pgrep" not in l]
done = lambda fam, job: (ROOT / f"{fam}__{job}" / "diagnostics.json").exists()

def start(fam, job):
    env = dict(os.environ, SENS_FAMILY=fam, SENS_JOB=job, OMP_NUM_THREADS="1", MKL_NUM_THREADS="1",
               OPENBLAS_NUM_THREADS="1", PYTHONUNBUFFERED="1", MPLBACKEND="Agg")
    log = open(LOGS / f"{fam}__{job}.log", "a"); log.write(f"\n===== start {time.strftime('%Y-%m-%d %H:%M:%S')} =====\n"); log.flush()
    subprocess.Popen([sys.executable, "/content/sens_worker.py"], stdout=log, stderr=subprocess.STDOUT,
                     env=env, cwd="/content", start_new_session=True)
    print("started", fam, job)

queue = [(f, j) for f, j in RUN_JOBS if not done(f, j)]
if running():
    print("workers already running here: monitoring only"); queue = []
while True:
    while queue and len(running()) < SLOTS:
        start(*queue.pop(0)); time.sleep(90)
    print(time.strftime("%H:%M:%S"), f"| load {os.getloadavg()[0]:.1f} | queued {len(queue)}")
    for f, j in RUN_JOBS:
        lf = LOGS / f"{f}__{j}.log"
        last = [l.rstrip() for l in lf.read_text(errors="ignore").splitlines() if l.startswith(f"{f}__{j}") or "Error" in l] if lf.exists() else []
        print(f"  {f + '__' + j:34s} {'DONE ' if done(f, j) else ''}{last[-1][len(f)+len(j)+4:][:120] if last else '(queued)'}")
    if not queue and not running():
        break
    time.sleep(600)
print("all jobs in this runtime finished; run cell 6")


In [ ]:
import json, zipfile, numpy as np, pandas as pd, arviz as az
from pathlib import Path
SENS = Path("/content/drive/MyDrive/new_dysfunction/normalized_state_bhm_ri_hc_v1")
ROOT = SENS / "_primary_sensitivity"; PRIM = ROOT / "_primary_summaries"; PRIM.mkdir(exist_ok=True)
ARCH = {"immune_coarse": "v2p2_g1_fixed_ctnu", "immune_fine": "v2p1c_g1_fixed_gnu", "nonimmune": "v2p1c_g1_fixed_gnu"}

def summ(da, dim, scale):
    da = da * scale
    h = az.hdi(da.to_dataset(name="v"), hdi_prob=0.95)["v"].transpose(dim, "hdi").values
    t = pd.DataFrame({dim: da[dim].values.astype(str), "mean": da.mean(("chain", "draw")).values,
                      "hdi95_low": h[:, 0], "hdi95_high": h[:, 1], "p_positive": (da > 0).mean(("chain", "draw")).values})
    t["excludes_zero"] = (t.hdi95_low > 0) | (t.hdi95_high < 0); return t

def primary(fam, per5):
    fc, cc = PRIM / f"{fam}_feature.csv", PRIM / f"{fam}_celltype.csv"
    if not fc.exists():
        tr = SENS / f"{ARCH[fam]}__{fam}__lowread1000__ri_hc_t" / "publication" / "posterior_trace.nc"
        post = az.from_netcdf(tr).posterior
        summ(post["feature_bmi_slope"], "feature", per5).to_csv(fc, index=False)
        summ(post["celltype_bmi_slope"], "celltype", per5).to_csv(cc, index=False)
    return pd.read_csv(fc), pd.read_csv(cc)

rows, ct_rows = [], []
for d in sorted(p for p in ROOT.iterdir() if p.is_dir() and (p / "diagnostics.json").exists()):
    r = json.loads((d / "diagnostics.json").read_text()); fam, job = r["family"], r["job"]
    pf, pc = primary(fam, r["per5_factor"])
    f = pd.read_csv(d / "feature_summary.csv"); c = pd.read_csv(d / "celltype_summary.csv")
    est = "per_5kg_m2" if "per_5kg_m2" in set(f.estimand) else "obese_vs_normal"
    m = f[f.estimand.eq(est)].merge(pf, on="feature", suffixes=("", "_p"))
    mc = c[c.estimand.eq(est)].merge(pc, on="celltype", suffixes=("", "_p"))
    w, wp = m.hdi95_high - m.hdi95_low, m.hdi95_high_p - m.hdi95_low_p
    rows.append({"family": fam, "job": job, "gate_pass": r["gate_pass"], "max_rhat": round(r["max_rhat"], 4),
                 "min_ess": int(min(r["min_ess_bulk"], r["min_ess_tail"])), "div": r["divergences"],
                 "estimand": est, "features_excl0": int(m.excludes_zero.sum()), "celltypes_excl0": int(mc.excludes_zero.sum()),
                 "primary_features_excl0": int(m.excludes_zero_p.sum()),
                 "r_feature_means_vs_primary": round(float(np.corrcoef(m["mean"], m.mean_p)[0, 1]), 3),
                 "median_width_ratio": round(float(np.median(w / wp)), 3) if est == "per_5kg_m2" else None,
                 "max_abs_celltype_shift": round(float((mc["mean"] - mc.mean_p).abs().max()), 4) if est == "per_5kg_m2" else None,
                 "celltype_sign_agreement": f"{int((np.sign(mc['mean']) == np.sign(mc.mean_p)).sum())}/{len(mc)}",
                 "minutes": round(r["minutes"])})
    for _, x in mc.iterrows():
        ct_rows.append({"family": fam, "job": job, "celltype": x.celltype.split("||")[-1], "estimand": est,
                        "mean": round(x["mean"], 4), "p_positive": round(x.p_positive, 3),
                        "primary_mean_per5": round(x.mean_p, 4), "primary_p_positive": round(x.p_positive_p, 3)})
tab = pd.DataFrame(rows); cts = pd.DataFrame(ct_rows)
pd.set_option("display.width", 250); print(tab.to_string(index=False))
tab.to_csv(ROOT / "sensitivity_vs_primary.csv", index=False); cts.to_csv(ROOT / "celltype_sensitivity_vs_primary.csv", index=False)
z = Path("/content/primary_sensitivity_results.zip")
with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as fz:
    for p in ROOT.rglob("*"):
        if p.is_file() and p.suffix in (".json", ".csv", ".log"):
            fz.write(p, p.relative_to(ROOT))
from google.colab import files; files.download(str(z))


In [ ]:
!pkill -f sens_worker.py; sleep 2; pgrep -fa sens_worker.py || echo "none running"
